# Accesibilidad QroBus hacia la estación Corregidora

Este notebook identifica:

1. **Paradas de origen** desde las que se puede llegar a la estación Corregidora en 30 minutos o menos.
2. **Itinerarios de rutas** que permiten llegar dentro del umbral, tanto directos como con transbordos.
3. Un mapa principal con capas consistentes y otro mapa con todas las paradas de 30 minutos que no requieren cruzar las vías.

El cálculo siempre se realiza **hacia Corregidora**. Los datos base provienen del GTFS local de QroBus en `data/`.


## 1. Metodología y alcance

Se construye un grafo dirigido con cada par consecutivo de paradas de los viajes GTFS. El tiempo de cada segmento es la mediana de los tiempos programados para la combinación `origen-destino-ruta`.

Para calcular caminos con transbordos se aplica Dijkstra sobre estados `(parada, ruta)`. Cambiar de ruta agrega una penalización configurable mediante `PENALIZACION_TRANSBORDO_MIN`.

> La penalización representa espera y caminata de conexión de forma aproximada. Un cálculo exacto necesita horarios por fecha, tiempos de espera y, preferentemente, GTFS-Realtime. El notebook no presenta esta estimación como telemetría real.

Si existe `data/tiempos_corregidos_google.csv`, se suma el atraso promedio disponible al tiempo de red. Si no existe, el análisis funciona únicamente con GTFS y lo indica explícitamente.


## 2. Correcciones realizadas durante QA

Se eliminaron:

- tres mapas parcialmente duplicados;
- análisis en sentido contrario, desde Corregidora hacia otras paradas;
- dependencias no necesarias como GeoPandas, Shapely, SciPy y Matplotlib;
- rutas geométricas construidas con un viaje representativo que podía pertenecer a la dirección incorrecta;
- bloques repetidos y capturas silenciosas de excepciones.

También se corrigió el modelo de transbordos: cambiar de ruta ya no tiene costo cero.


## 3. Configuración

Los parámetros se leen desde `.env`:

```dotenv
UMBRAL_ANALISIS_MIN=30
PENALIZACION_TRANSBORDO_MIN=5
```

`RutasQroBus.ipynb` no llama a Google Maps. Solo consume la corrección ya generada por `PromedioRutas.ipynb`, cuando está disponible.


In [1]:
from pathlib import Path
import os
import sys

import folium
import numpy as np
import pandas as pd
from IPython.display import display

# Funciona al ejecutar desde scripts/ o desde la raíz del repositorio.
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
if not DATA_DIR.exists():
    raise FileNotFoundError("No se encontró la carpeta local data/.")

PROJECT_ROOT = DATA_DIR.parent.resolve()
ENV_PATH = PROJECT_ROOT / ".env"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from gtfs_network import (
    cargar_rutas_base,
    compactar_rutas,
    gtfs_time_to_seconds,
    VIA_FERREA_CORREGIDORA_LATLON,
)


def cargar_env(env_path):
    """Carga pares CLAVE=VALOR sencillos sin imprimir secretos."""
    if not env_path.exists():
        return False

    for numero_linea, linea_original in enumerate(
        env_path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        linea = linea_original.strip()
        if not linea or linea.startswith("#"):
            continue
        if linea.startswith("export "):
            linea = linea[7:].strip()
        if "=" not in linea:
            raise ValueError(f"Línea inválida en .env: {numero_linea}")

        clave, valor = linea.split("=", 1)
        clave, valor = clave.strip(), valor.strip()
        if len(valor) >= 2 and valor[0] == valor[-1] and valor[0] in {"'", '"'}:
            valor = valor[1:-1]
        os.environ.setdefault(clave, valor)

    return True


ENV_CARGADO = cargar_env(ENV_PATH)
UMBRAL_ANALISIS_MIN = float(os.getenv("UMBRAL_ANALISIS_MIN", "30"))
PENALIZACION_TRANSBORDO_MIN = float(
    os.getenv("PENALIZACION_TRANSBORDO_MIN", "5")
)

if UMBRAL_ANALISIS_MIN <= 0:
    raise ValueError("UMBRAL_ANALISIS_MIN debe ser mayor que cero.")
if PENALIZACION_TRANSBORDO_MIN < 0:
    raise ValueError("PENALIZACION_TRANSBORDO_MIN no puede ser negativa.")

CORREGIDORA_LAT = 20.600611
CORREGIDORA_LON = -100.402184

print(f"Datos: {DATA_DIR.resolve()}")
print(f".env cargado: {ENV_CARGADO}")
print(f"Umbral: {UMBRAL_ANALISIS_MIN:g} min")
print(f"Penalización por transbordo: {PENALIZACION_TRANSBORDO_MIN:g} min")


Datos: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data
.env cargado: True
Umbral: 30 min
Penalización por transbordo: 5 min


## 4. Lectura y validación de paradas

Los identificadores se cargan como texto para conservar valores como `005`. Este notebook solo necesita `stops.txt` para dibujar el mapa; el resto del GTFS ya fue procesado por `PromedioRutas.ipynb`.


In [2]:
stops = pd.read_csv(DATA_DIR / "stops.txt", dtype={"stop_id": "string"})
columnas_requeridas = {"stop_id", "stop_name", "stop_lat", "stop_lon"}
faltantes = columnas_requeridas - set(stops.columns)
if faltantes:
    raise ValueError(f"stops.txt no contiene: {sorted(faltantes)}")

if stops["stop_id"].duplicated().any():
    raise ValueError("stops.txt contiene stop_id duplicados.")
for columna in ["stop_lat", "stop_lon"]:
    stops[columna] = pd.to_numeric(stops[columna], errors="raise")

print(f"{len(stops):,} paradas cargadas para construir el mapa.")


2,694 paradas cargadas para construir el mapa.


## 5. Carga del cálculo base

`PromedioRutas.ipynb` ya convirtió los horarios GTFS, construyó los segmentos y ejecutó Dijkstra con transbordos. Aquí se carga `data/rutas_base_gtfs.csv`, se restauran las listas que forman cada camino y se valida que haya sido creado con la misma penalización configurada.


In [3]:
RUTAS_BASE_PATH = DATA_DIR / "rutas_base_gtfs.csv"
if not RUTAS_BASE_PATH.exists():
    raise FileNotFoundError(
        "Falta data/rutas_base_gtfs.csv. Ejecute PromedioRutas.ipynb primero."
    )

resultados_base = cargar_rutas_base(RUTAS_BASE_PATH)
columnas_base_requeridas = {
    "stop_id",
    "tiempo_red_min",
    "num_transbordos",
    "tipo_conexion",
    "itinerario_route_ids",
    "itinerario_rutas",
    "paradas_transbordo",
    "camino_stop_ids",
    "rutas_por_segmento",
    "penalizacion_transbordo_config_min",
}
faltantes = columnas_base_requeridas - set(resultados_base.columns)
if faltantes:
    raise ValueError(f"El cálculo base no contiene: {sorted(faltantes)}")

penalizaciones_base = pd.to_numeric(
    resultados_base["penalizacion_transbordo_config_min"], errors="raise"
).dropna().unique()
if len(penalizaciones_base) != 1 or not np.isclose(
    penalizaciones_base[0], PENALIZACION_TRANSBORDO_MIN
):
    raise ValueError(
        "La penalización del cálculo base no coincide con .env. "
        "Vuelva a ejecutar PromedioRutas.ipynb."
    )

print(
    f"Cálculo base cargado: {len(resultados_base):,} paradas; "
    "Dijkstra no se vuelve a ejecutar en este notebook."
)


Cálculo base cargado: 2,259 paradas; Dijkstra no se vuelve a ejecutar en este notebook.


## 6. Destino: estación Corregidora

Las coordenadas del proyecto se comparan con todas las paradas mediante distancia Haversine. El nodo destino del grafo es la parada QroBus más cercana. El marcador del mapa conserva las coordenadas exactas de la estación.


In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    radio_tierra_km = 6371.0088
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = (
        np.sin(delta_phi / 2) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2) ** 2
    )
    return radio_tierra_km * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


distancias_estacion = haversine_km(
    stops["stop_lat"].to_numpy(),
    stops["stop_lon"].to_numpy(),
    CORREGIDORA_LAT,
    CORREGIDORA_LON,
)
target_idx = int(np.argmin(distancias_estacion))
target_row = stops.iloc[target_idx]
corregidora_stop_id = target_row["stop_id"]
distancia_estacion_stop_m = float(distancias_estacion[target_idx] * 1000)

print(
    f"Parada destino: {corregidora_stop_id} — {target_row['stop_name']} "
    f"({distancia_estacion_stop_m:.0f} m de las coordenadas de la estación)."
)


Parada destino: 3025 — Estío/Calle Dr. Manuel Domínguez (211 m de las coordenadas de la estación).


## 7. Reutilización de los caminos mínimos

Los caminos mínimos con transbordos vienen completos desde `rutas_base_gtfs.csv`. Se verifica que el destino guardado sea la parada GTFS más cercana a Corregidora y después se continúa directamente con las correcciones de Google y la visualización.


In [5]:
resultados = resultados_base.copy()
destinos_base = resultados[resultados["tipo_conexion"].eq("Destino")]
if len(destinos_base) != 1:
    raise ValueError("El cálculo base debe contener exactamente un destino.")
if destinos_base.iloc[0]["stop_id"] != corregidora_stop_id:
    raise ValueError(
        "El destino del cálculo base no coincide con Corregidora. "
        "Vuelva a ejecutar PromedioRutas.ipynb."
    )

print(f"Paradas alcanzables cargadas: {len(resultados):,}")


Paradas alcanzables cargadas: 2,259


In [6]:
display(
    resultados[
        [
            "stop_id",
            "stop_name",
            "tiempo_red_min",
            "num_transbordos",
            "itinerario_rutas",
        ]
    ].head(10)
)


,stop_id,stop_name,tiempo_red_min,num_transbordos,itinerario_rutas
0,3025,Estío/Calle Dr. Manuel Domínguez,0.000000,0,NaN
1,2341,Av. Felipe Ángeles/Av. San Roque,1.533333,0,C54
2,3295,Felipe Ángeles/Fraternidad,2.216667,0,C54
3,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,2.733333,0,C54
4,3294,Av. Felipe Ángeles/Calle del Porvenir,3.583333,0,C54
5,2339,Av. Felipe Ángeles/Felipe Ángeles 225,4.200000,0,C54
6,2338,Av. Felipe Ángeles/Estadística,4.700000,0,C54
7,4256,Epigmenio González/Departamental Parques,5.366667,0,C54
8,3363,Prol. Tecnológico/Calle San Joaquín,8.033333,0,C54
9,3362,Prol. Tecnológico/Ford Citelis Querétaro,9.200000,0,C54


## 8. Corrección opcional con observaciones de Google

La corrección se aplica únicamente cuando la fila está marcada como disponible. Se suma `atraso_promedio_ruta_min` al tiempo de red calculado aquí; no se reemplaza el camino ni se modifica el número de transbordos.

Las filas sin observación continúan como `GTFS programado`. Esta separación evita presentar datos no observados como si provinieran de Google.


In [7]:
correcciones_path = DATA_DIR / "tiempos_corregidos_google.csv"

if correcciones_path.exists():
    correcciones = pd.read_csv(
        correcciones_path, dtype={"stop_id": "string"}
    )
    requeridas = {
        "stop_id",
        "atraso_promedio_ruta_min",
        "correccion_disponible",
    }
    faltantes = requeridas - set(correcciones.columns)
    if faltantes:
        raise ValueError(
            f"El archivo de correcciones no contiene: {sorted(faltantes)}"
        )

    correcciones = correcciones[
        [
            "stop_id",
            "atraso_promedio_ruta_min",
            "correccion_disponible",
        ]
    ].drop_duplicates("stop_id", keep="last")

    disponible = correcciones["correccion_disponible"]
    correcciones["correccion_disponible"] = (
        disponible.eq(True)
        | disponible.astype(str).str.lower().eq("true")
    )
    correcciones["atraso_promedio_ruta_min"] = pd.to_numeric(
        correcciones["atraso_promedio_ruta_min"], errors="coerce"
    )

    resultados = resultados.merge(
        correcciones,
        on="stop_id",
        how="left",
        validate="one_to_one",
    )
else:
    resultados["atraso_promedio_ruta_min"] = np.nan
    resultados["correccion_disponible"] = False

resultados["correccion_disponible"] = resultados[
    "correccion_disponible"
].fillna(False).astype(bool)
resultados["atraso_aplicado_min"] = (
    resultados["atraso_promedio_ruta_min"]
    .where(resultados["correccion_disponible"], 0)
    .fillna(0)
    .clip(lower=0)
)
resultados["tiempo_estimado_min"] = (
    resultados["tiempo_red_min"] + resultados["atraso_aplicado_min"]
)
resultados["fuente_tiempo"] = np.where(
    resultados["correccion_disponible"],
    "GTFS + corrección Google",
    "GTFS programado",
)

cobertura_google = resultados.loc[
    resultados["stop_id"].ne(corregidora_stop_id),
    "correccion_disponible",
].mean()

print(
    f"Corrección Google encontrada: {correcciones_path.exists()}. "
    f"Cobertura de paradas alcanzables: {cobertura_google:.1%}."
)


Corrección Google encontrada: True. Cobertura de paradas alcanzables: 86.0%.


## 9. Resultados dentro de 30 minutos

La parada destino no se cuenta como origen. Una parada se clasifica dentro del umbral usando `tiempo_estimado_min`, que incluye:

- tiempo mediano de segmentos GTFS;
- penalización por cada transbordo;
- atraso de Google, solo cuando está disponible.


In [8]:
paradas_30 = resultados[
    resultados["stop_id"].ne(corregidora_stop_id)
    & resultados["tiempo_estimado_min"].le(UMBRAL_ANALISIS_MIN)
].copy()

paradas_directas_30 = paradas_30[
    paradas_30["num_transbordos"].eq(0)
].copy()
paradas_con_transbordo_30 = paradas_30[
    paradas_30["num_transbordos"].gt(0)
].copy()

tabla_itinerarios_30 = (
    paradas_30.groupby(
        [
            "itinerario_route_ids",
            "itinerario_rutas",
            "tipo_conexion",
            "num_transbordos",
        ],
        as_index=False,
    )
    .agg(
        paradas_origen=("stop_id", "nunique"),
        tiempo_minimo_min=("tiempo_estimado_min", "min"),
        tiempo_maximo_min=("tiempo_estimado_min", "max"),
        tiempo_promedio_min=("tiempo_estimado_min", "mean"),
    )
    .sort_values(
        ["num_transbordos", "tiempo_minimo_min", "itinerario_rutas"]
    )
    .reset_index(drop=True)
)

print(f"Paradas de origen dentro de {UMBRAL_ANALISIS_MIN:g} min: {len(paradas_30):,}")
print(f"  Conexión directa: {len(paradas_directas_30):,}")
print(f"  Con transbordos: {len(paradas_con_transbordo_30):,}")
print(f"Itinerarios de rutas distintos: {len(tabla_itinerarios_30):,}")


Paradas de origen dentro de 30 min: 116
  Conexión directa: 15
  Con transbordos: 101
Itinerarios de rutas distintos: 24


### 9.1 Rutas e itinerarios que llegan dentro del umbral

Cada fila representa una secuencia de rutas, no una geometría duplicada por cada parada. `C54` indica un viaje directo; `CXX → C54` indica un transbordo.


In [9]:
columnas_itinerarios = [
    "itinerario_rutas",
    "tipo_conexion",
    "num_transbordos",
    "paradas_origen",
    "tiempo_minimo_min",
    "tiempo_promedio_min",
    "tiempo_maximo_min",
]
display(
    tabla_itinerarios_30[columnas_itinerarios].style.format(
        {
            "tiempo_minimo_min": "{:.1f}",
            "tiempo_promedio_min": "{:.1f}",
            "tiempo_maximo_min": "{:.1f}",
        }
    )
)


,itinerario_rutas,tipo_conexion,num_transbordos,paradas_origen,tiempo_minimo_min,tiempo_promedio_min,tiempo_maximo_min
0,C54,Directa,0,15,6.7,13.3,20.4
1,C33 → C54,Con transbordo,1,20,15.0,22.1,29.4
2,T07 → C54,Con transbordo,1,6,15.5,20.4,24.7
3,L154 → C54,Con transbordo,1,6,18.1,21.8,26.1
4,C62 → C54,Con transbordo,1,8,22.4,26.8,30.0
5,C56 → C54,Con transbordo,1,6,24.9,27.2,29.8
6,T03 → C54,Con transbordo,1,3,27.7,28.6,29.4
7,C63 → T07 → C54,Con transbordo,2,2,22.4,23.0,23.7
8,C60 → C33 → C54,Con transbordo,2,5,23.3,26.2,29.3
9,C24 → C33 → C54,Con transbordo,2,5,23.4,27.1,29.6


### 9.2 Paradas dentro del umbral, incluyendo transbordos

La tabla conserva el itinerario, los puntos de transbordo y la fuente del tiempo. Esto permite distinguir estimaciones GTFS de aquellas que ya incorporan una corrección de Google.


In [10]:
tabla_paradas_30 = paradas_30[
    [
        "stop_id",
        "stop_name",
        "tipo_conexion",
        "itinerario_rutas",
        "paradas_transbordo",
        "num_transbordos",
        "num_paradas_camino",
        "tiempo_red_min",
        "atraso_aplicado_min",
        "tiempo_estimado_min",
        "fuente_tiempo",
    ]
].sort_values(["tiempo_estimado_min", "stop_name"])

display(
    tabla_paradas_30.style.format(
        {
            "tiempo_red_min": "{:.1f}",
            "atraso_aplicado_min": "{:.1f}",
            "tiempo_estimado_min": "{:.1f}",
        }
    )
)


,stop_id,stop_name,tipo_conexion,itinerario_rutas,paradas_transbordo,num_transbordos,num_paradas_camino,tiempo_red_min,atraso_aplicado_min,tiempo_estimado_min,fuente_tiempo
1,2341,Av. Felipe Ángeles/Av. San Roque,Directa,C54,nan,0,2,1.5,5.2,6.7,GTFS + corrección Google
2,3295,Felipe Ángeles/Fraternidad,Directa,C54,nan,0,3,2.2,5.2,7.4,GTFS + corrección Google
3,2340,Av. Felipe Ángeles/Plan de Ayala Poniente,Directa,C54,nan,0,4,2.7,5.2,7.9,GTFS + corrección Google
4,3294,Av. Felipe Ángeles/Calle del Porvenir,Directa,C54,nan,0,5,3.6,5.2,8.8,GTFS + corrección Google
5,2339,Av. Felipe Ángeles/Felipe Ángeles 225,Directa,C54,nan,0,6,4.2,5.2,9.4,GTFS + corrección Google
6,2338,Av. Felipe Ángeles/Estadística,Directa,C54,nan,0,7,4.7,5.2,9.9,GTFS + corrección Google
7,4256,Epigmenio González/Departamental Parques,Directa,C54,nan,0,8,5.4,5.2,10.6,GTFS + corrección Google
8,3363,Prol. Tecnológico/Calle San Joaquín,Directa,C54,nan,0,9,8.0,5.2,13.2,GTFS + corrección Google
9,3362,Prol. Tecnológico/Ford Citelis Querétaro,Directa,C54,nan,0,10,9.2,5.2,14.4,GTFS + corrección Google
13,3284,Epigmenio González/Prol. Porvenir,Con transbordo,C33 → C54,Epigmenio González/Departamental Parques,1,9,12.7,2.3,15.0,GTFS + corrección Google


## 10. Pruebas automáticas de calidad

Estas comprobaciones validan que los caminos terminen en Corregidora, que los tiempos sean no negativos, que los transbordos coincidan con los cambios de ruta y que ninguna parada del reporte exceda el umbral.


In [11]:
assert corregidora_stop_id in set(resultados["stop_id"])
assert resultados["stop_id"].is_unique
assert np.isfinite(resultados["tiempo_red_min"]).all()
assert resultados["tiempo_red_min"].ge(0).all()
assert resultados["atraso_aplicado_min"].ge(0).all()
assert paradas_30["tiempo_estimado_min"].le(
    UMBRAL_ANALISIS_MIN + 1e-9
).all()
assert paradas_30["stop_id"].ne(corregidora_stop_id).all()

for fila in resultados.itertuples(index=False):
    assert fila.camino_stop_ids[-1] == corregidora_stop_id
    rutas_compactas = compactar_rutas(fila.rutas_por_segmento)
    assert fila.num_transbordos == max(len(rutas_compactas) - 1, 0)
    assert len(fila.camino_stop_ids) == len(fila.rutas_por_segmento) + 1

print("QA OK: todas las validaciones automáticas fueron superadas.")


QA OK: todas las validaciones automáticas fueron superadas.


## 11. Mapa principal de accesibilidad

El mapa consolida las visualizaciones anteriores:

- **Azul:** paradas con conexión directa.
- **Naranja:** paradas que requieren uno o más transbordos.
- **Rojo:** estación Corregidora y su parada GTFS más cercana.
- **Trayectos representativos:** un camino por cada itinerario de rutas, para evitar cientos de líneas duplicadas.

El control de capas permite ocultar o mostrar cada grupo sin crear mapas inconsistentes.


In [12]:
mapa_30 = folium.Map(
    location=[CORREGIDORA_LAT, CORREGIDORA_LON],
    zoom_start=13,
    tiles="OpenStreetMap",
)

capa_directas = folium.FeatureGroup(
    name=f"Paradas directas ({len(paradas_directas_30)})",
    show=True,
)
capa_transbordos = folium.FeatureGroup(
    name=f"Paradas con transbordo ({len(paradas_con_transbordo_30)})",
    show=True,
)
capa_trayectos = folium.FeatureGroup(
    name=f"Trayectos representativos ({len(tabla_itinerarios_30)})",
    show=True,
)

folium.Marker(
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    tooltip="Estación Corregidora",
    popup=(
        f"<b>Estación Corregidora</b><br>"
        f"Parada GTFS cercana: {target_row['stop_name']}<br>"
        f"Distancia: {distancia_estacion_stop_m:.0f} m"
    ),
    icon=folium.Icon(color="red", icon="flag"),
).add_to(mapa_30)

folium.CircleMarker(
    [target_row["stop_lat"], target_row["stop_lon"]],
    radius=5,
    color="darkred",
    fill=True,
    fill_opacity=0.9,
    tooltip=f"Parada destino GTFS: {target_row['stop_name']}",
).add_to(mapa_30)

for fila in paradas_directas_30.itertuples(index=False):
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon],
        radius=4,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.75,
        tooltip=(
            f"{fila.stop_name} | {fila.tiempo_estimado_min:.1f} min | "
            f"{fila.itinerario_rutas}"
        ),
    ).add_to(capa_directas)

for fila in paradas_con_transbordo_30.itertuples(index=False):
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon],
        radius=4,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=0.8,
        tooltip=(
            f"{fila.stop_name} | {fila.tiempo_estimado_min:.1f} min | "
            f"{fila.itinerario_rutas} | "
            f"{fila.num_transbordos} transbordo(s)"
        ),
    ).add_to(capa_transbordos)

representantes = (
    paradas_30.sort_values("tiempo_estimado_min")
    .drop_duplicates("itinerario_route_ids", keep="first")
)

coords_map = stops.set_index("stop_id")[
    ["stop_lat", "stop_lon"]
].to_dict("index")

for fila in representantes.itertuples(index=False):
    coordenadas_camino = [
        (
            coords_map[stop_id]["stop_lat"],
            coords_map[stop_id]["stop_lon"],
        )
        for stop_id in fila.camino_stop_ids
    ]
    color = "blue" if fila.num_transbordos == 0 else "orange"
    folium.PolyLine(
        coordenadas_camino,
        color=color,
        weight=3,
        opacity=0.65,
        tooltip=(
            f"{fila.itinerario_rutas} | "
            f"{fila.tiempo_estimado_min:.1f} min"
        ),
    ).add_to(capa_trayectos)

capa_directas.add_to(mapa_30)
capa_transbordos.add_to(mapa_30)
capa_trayectos.add_to(mapa_30)
folium.LayerControl(collapsed=False).add_to(mapa_30)

coordenadas_visibles = [
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    *paradas_30[["stop_lat", "stop_lon"]].to_numpy().tolist(),
]
if len(coordenadas_visibles) > 1:
    mapa_30.fit_bounds(coordenadas_visibles)

MAPA_PATH = DATA_DIR / "mapa_rutas_qrobus.html"
mapa_30.save(MAPA_PATH)
print(f"Mapa interactivo exportado: {MAPA_PATH.resolve()}")

# Al quedar como la última expresión, Jupyter integra el mapa en la salida de la celda.
# No depende del archivo HTML exportado arriba.
mapa_30


Mapa interactivo exportado: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/mapa_rutas_qrobus.html


## 12. Todas las paradas a menos de 30 minutos sin cruzar las vías

Este escenario carga el segundo Dijkstra generado por `PromedioRutas.ipynb` después de retirar del grafo los segmentos que cruzan la `Línea Juárez`. La parada de Mariano Escobedo solo define el lado sur de referencia; el resultado incluye **todas** las paradas que todavía pueden llegar a Corregidora dentro del umbral mediante autobús y un último tramo a pie de diez minutos o menos.

Para no reutilizar una corrección asociada a una ruta distinta, el atraso se vuelve a unir por la ruta principal del nuevo camino. La polilínea ferroviaria procede de OpenStreetMap; datos cartográficos © [colaboradores de OpenStreetMap](https://www.openstreetmap.org/copyright), ODbL.


In [13]:
RUTAS_SUR_VIAS_PATH = DATA_DIR / "rutas_base_sur_vias.csv"
DESTINO_PEATONAL_SUR_ID = "ESTACION_CORREGIDORA_SUR"
if not RUTAS_SUR_VIAS_PATH.exists():
    raise FileNotFoundError(
        "Falta data/rutas_base_sur_vias.csv. Ejecute PromedioRutas.ipynb primero."
    )

resultados_sur_vias = cargar_rutas_base(RUTAS_SUR_VIAS_PATH)
destino_sur = resultados_sur_vias[
    resultados_sur_vias["tipo_conexion"].eq("Destino")
]
if (
    len(destino_sur) != 1
    or destino_sur.iloc[0]["stop_id"] != DESTINO_PEATONAL_SUR_ID
):
    raise ValueError(
        "El escenario sin cruces no corresponde al destino actual. "
        "Vuelva a ejecutar PromedioRutas.ipynb."
    )
penalizacion_sur = pd.to_numeric(
    resultados_sur_vias["penalizacion_transbordo_config_min"],
    errors="raise",
).dropna().unique()
if len(penalizacion_sur) != 1 or not np.isclose(
    penalizacion_sur[0], PENALIZACION_TRANSBORDO_MIN
):
    raise ValueError(
        "La penalización del escenario sin cruces no coincide con .env."
    )

PROMEDIO_RUTAS_PATH = DATA_DIR / "promedio_atrasos_google_por_ruta.csv"
resultados_sur_vias["route_id_union"] = pd.to_numeric(
    resultados_sur_vias["route_id_principal"], errors="coerce"
)
if PROMEDIO_RUTAS_PATH.exists():
    atrasos_ruta_sur = pd.read_csv(PROMEDIO_RUTAS_PATH)
    columnas_atraso = {
        "route_id_principal",
        "atraso_promedio_ruta_min",
    }
    faltantes = columnas_atraso - set(atrasos_ruta_sur.columns)
    if faltantes:
        raise ValueError(f"El promedio por ruta no contiene: {sorted(faltantes)}")
    atrasos_ruta_sur["route_id_union"] = pd.to_numeric(
        atrasos_ruta_sur["route_id_principal"], errors="coerce"
    )
    atrasos_ruta_sur["atraso_promedio_ruta_min"] = pd.to_numeric(
        atrasos_ruta_sur["atraso_promedio_ruta_min"], errors="coerce"
    )
    resultados_sur_vias = resultados_sur_vias.merge(
        atrasos_ruta_sur[
            ["route_id_union", "atraso_promedio_ruta_min"]
        ].drop_duplicates("route_id_union"),
        on="route_id_union",
        how="left",
        validate="many_to_one",
    )
else:
    resultados_sur_vias["atraso_promedio_ruta_min"] = np.nan

resultados_sur_vias["correccion_disponible"] = resultados_sur_vias[
    "atraso_promedio_ruta_min"
].notna()
resultados_sur_vias["atraso_aplicado_min"] = resultados_sur_vias[
    "atraso_promedio_ruta_min"
].fillna(0).clip(lower=0)
resultados_sur_vias["tiempo_estimado_min"] = (
    resultados_sur_vias["tiempo_red_min"]
    + resultados_sur_vias["atraso_aplicado_min"]
)
resultados_sur_vias["fuente_tiempo"] = np.where(
    resultados_sur_vias["correccion_disponible"],
    "GTFS + corrección Google por ruta",
    "GTFS programado",
)

paradas_sur_vias_30 = resultados_sur_vias[
    resultados_sur_vias["stop_id"].ne(DESTINO_PEATONAL_SUR_ID)
    & resultados_sur_vias["tiempo_estimado_min"].le(UMBRAL_ANALISIS_MIN)
].sort_values(["tiempo_estimado_min", "stop_name"]).copy()
if "2312" not in set(paradas_sur_vias_30["stop_id"]):
    raise RuntimeError(
        "La parada de Mariano Escobedo no quedó dentro del escenario de 30 minutos."
    )

print(
    f"Paradas al sur de las vías dentro de {UMBRAL_ANALISIS_MIN:g} min: "
    f"{len(paradas_sur_vias_30):,}."
)
print(
    "La parada 2312 solo se utilizó para elegir el lado de la vía; "
    "no limita los orígenes del resultado."
)
columnas_sur_vias = [
    "stop_id",
    "stop_name",
    "tipo_conexion",
    "itinerario_rutas",
    "num_transbordos",
    "tiempo_red_min",
    "atraso_aplicado_min",
    "tiempo_estimado_min",
    "fuente_tiempo",
]
display(
    paradas_sur_vias_30[columnas_sur_vias].style.format(
        {
            "tiempo_red_min": "{:.1f}",
            "atraso_aplicado_min": "{:.1f}",
            "tiempo_estimado_min": "{:.1f}",
        }
    )
)

mapa_sur_vias = folium.Map(
    location=[CORREGIDORA_LAT, CORREGIDORA_LON],
    zoom_start=13,
    tiles="OpenStreetMap",
)
folium.PolyLine(
    VIA_FERREA_CORREGIDORA_LATLON,
    color="black",
    weight=5,
    opacity=0.85,
    tooltip="Línea ferroviaria usada como barrera",
).add_to(mapa_sur_vias)
folium.Marker(
    [CORREGIDORA_LAT, CORREGIDORA_LON],
    tooltip="Estación Corregidora",
    icon=folium.Icon(color="red", icon="train", prefix="fa"),
).add_to(mapa_sur_vias)

for fila in paradas_sur_vias_30.itertuples(index=False):
    es_referencia = fila.stop_id == "2312"
    folium.CircleMarker(
        [fila.stop_lat, fila.stop_lon],
        radius=6 if es_referencia else 4,
        color="green" if es_referencia else "teal",
        fill=True,
        fill_opacity=0.8,
        tooltip=(
            f"{fila.stop_name} | {fila.tiempo_estimado_min:.1f} min | "
            f"{fila.itinerario_rutas}"
        ),
    ).add_to(mapa_sur_vias)

representantes_sur = (
    paradas_sur_vias_30.sort_values("tiempo_estimado_min")
    .drop_duplicates("itinerario_route_ids", keep="first")
)
coords_sur = resultados_sur_vias.set_index("stop_id")[
    ["stop_lat", "stop_lon"]
].to_dict("index")
for fila in representantes_sur.itertuples(index=False):
    coordenadas = [
        [coords_sur[stop_id]["stop_lat"], coords_sur[stop_id]["stop_lon"]]
        for stop_id in fila.camino_stop_ids
    ]
    folium.PolyLine(
        coordenadas,
        color="teal",
        weight=3,
        opacity=0.55,
        tooltip=f"{fila.itinerario_rutas} | {fila.tiempo_estimado_min:.1f} min",
    ).add_to(mapa_sur_vias)

mapa_sur_vias.fit_bounds(
    [
        [CORREGIDORA_LAT, CORREGIDORA_LON],
        *paradas_sur_vias_30[["stop_lat", "stop_lon"]].to_numpy().tolist(),
    ]
)
MAPA_SUR_VIAS_PATH = DATA_DIR / "mapa_paradas_sur_vias_30_min.html"
mapa_sur_vias.save(MAPA_SUR_VIAS_PATH)
print(f"Mapa sin cruces exportado: {MAPA_SUR_VIAS_PATH.resolve()}")
mapa_sur_vias


Paradas al sur de las vías dentro de 30 min: 234.
La parada 2312 solo se utilizó para elegir el lado de la vía; no limita los orígenes del resultado.


,stop_id,stop_name,tipo_conexion,itinerario_rutas,num_transbordos,tiempo_red_min,atraso_aplicado_min,tiempo_estimado_min,fuente_tiempo
1,3025,Estío/Calle Dr. Manuel Domínguez,Solo caminata,nan,0,2.6,0.0,2.6,GTFS programado
2,5200,Calle Nicolás Bravo/Calle Primavera,Solo caminata,nan,0,4.4,0.0,4.4,GTFS programado
3,3263,San Agustín del Retablo/Salvador Galván,Solo caminata,nan,0,4.8,0.0,4.8,GTFS programado
7,5193,San Agustín del Retablo/Estio,Solo caminata,nan,0,5.6,0.0,5.6,GTFS programado
9,2680,Av. Universidad/Calle Ezequiel Montes,Solo caminata,nan,0,5.8,0.0,5.8,GTFS programado
12,3283,José Amilcar Vidal/San Agustín del Retablo,Solo caminata,nan,0,6.3,0.0,6.3,GTFS programado
4,2316,Calle Nicolás Bravo/Calle Segunda de Las Rosas,Directa,C55,0,5.1,1.7,6.8,GTFS + corrección Google por ruta
14,2957,Av. Universidad/Tecnológico Nacional de México Campus Ciidet,Solo caminata,nan,0,6.8,0.0,6.8,GTFS programado
5,5194,Av. Universidad/Calle Rafael Osuna,Directa,C55,0,5.5,1.7,7.3,GTFS + corrección Google por ruta
10,2315,Av. Universidad/San Andrés,Directa,C55,0,6.0,1.7,7.7,GTFS + corrección Google por ruta


Mapa sin cruces exportado: /Users/manuelbajos/Documents/tec/semestres/Septimo/ProyectoInvestigacion/github/Tren_Mex_Qro/data/mapa_paradas_sur_vias_30_min.html


## 13. Interpretación correcta

El resultado responde: **“¿Qué paradas tienen un camino programado hacia Corregidora cuyo tiempo mediano, penalizaciones de transbordo y corrección disponible no superan 30 minutos?”**

No responde todavía:

- cuál unidad específica llegará;
- si un transbordo concreto estará sincronizado;
- el atraso GPS real del autobús;
- qué servicio opera en una fecha particular, porque el conjunto local no incluye `calendar.txt` ni `calendar_dates.txt`.

Para operación en tiempo real debe incorporarse GTFS-Realtime `TripUpdates` y, de ser posible, `VehiclePositions`.
